# Family Relations Ontology

Yeh example [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners) ka hissa hai, aur ise [is blog post](https://habr.com/post/270857/) se inspire hokar banaya gaya hai.

Mujhe hamesha family mein logon ke beech ke alag-alag relations yaad rakhna mushkil lagta hai. Is example mein, hum ek ontology lenge jo family relations define karti hai, aur ek real genealogical tree use karenge, aur phir dikhayenge ki hum automatic inference karke saare relatives kaise dhoondh sakte hain.

### Genealogical tree paana

Example ke liye, hum [Romanov Tsar family](https://en.wikipedia.org/wiki/House_of_Romanov) ka genealogical tree lenge. Family relations describe karne ka sabse common format [GEDCOM](https://en.wikipedia.org/wiki/GEDCOM) hai. Hum GEDCOM format mein Romanov family ka genealogical tree lenge:

In [ ]:
!head -15 data/tsars.ged

GEDCOM file use karne ke liye, hum `python-gedcom` library use kar sakte hain:

In [ ]:
import sys
!{sys.executable} -m pip install python-gedcom

Yeh library file parsing se jude kuch technical issues door kar deti hai, lekin yeh abhi bhi humein tree ke saare persons aur families tak kaafi low-level access deti hai. Yahan dekhiye ki hum file ko kaise parse kar sakte hain aur saare persons ki list kaise dikha sakte hain:

In [ ]:
from gedcom.parser import Parser
from gedcom.element.individual import IndividualElement
from gedcom.element.family import FamilyElement
g = Parser()
g.parse_file('data/tsars.ged')

In [ ]:
d = g.get_element_dictionary()
[ (k,v.get_name()) for k,v in d.items() if isinstance(v,IndividualElement)]

Yahan dekhiye ki hum families ke baare mein information kaise paa sakte hain. Dhyan dein ki yeh humein **identifiers** ki ek list deta hai, aur agar humein zyada clarity chahiye toh unhe names mein convert karna hoga:

In [ ]:
d = g.get_element_dictionary()
[ (k,[x.get_value() for x in v.get_child_elements()]) for k,v in d.items() if isinstance(v,FamilyElement)]

### Family ontology paana

Ab, chalo [family ontology](https://raw.githubusercontent.com/blokhin/genealogical-trees/master/data/header.ttl) par nazar daalte hain, jise semantic web triplets ke set ke roop mein define kiya gaya hai. Is ontology mein `isUncleOf`, `isCousinOf` aur kai aur relations define hain. Yeh saare relations basic predicates `isMotherOf`, `isFatherOf`, `isBrotherOf` aur `isSisterOf` ke base par define kiye gaye hain. Hum ontology use karke automatic reasoning ke zariye baaki saare relations nikaalenge.

Yahan `isAuntOf` property ki ek sample definition hai, jise `isSisterOf` aur `isParentOf` ke combination ke roop mein define kiya gaya hai (*aunt kisi ke parent ki sister hoti hai*).

```
fhkb:isAuntOf a owl:ObjectProperty ;
    rdfs:domain fhkb:Woman ;
    rdfs:range fhkb:Person ;
    owl:propertyChainAxiom ( fhkb:isSisterOf fhkb:isParentOf ) .
```

In [ ]:
!head -20 data/onto.ttl

### Inference ke liye ontology banana

Simplicity ke liye, hum ek ontology file banayenge jisme family ontology ke basic rules aur hamari GEDCOM file se persons ke baare mein facts shamil honge. Hum GEDCOM file ko analyze karenge, families aur persons ki information nikaalenge, aur unhe triplets mein convert karenge.

In [ ]:
!cp data/onto.ttl .

gedcom_dict = g.get_element_dictionary()
individuals, marriages = {}, {}

def term2id(el):
    return "i" + el.get_pointer().replace('@', '').lower()

out = open("onto.ttl","a")

for k, v in gedcom_dict.items():
    if isinstance(v,IndividualElement):
        children, siblings = set(), set()
        idx = term2id(v)

        title = v.get_name()[0] + " " + v.get_name()[1]
        title = title.replace('"', '').replace('[', '').replace(']', '').replace('(', '').replace(')', '').strip()

        own_families = g.get_families(v, 'FAMS')
        for fam in own_families:
            children |= set(term2id(i) for i in g.get_family_members(fam, "CHIL"))

        parent_families = g.get_families(v, 'FAMC')
        if len(parent_families):
            for member in g.get_family_members(parent_families[0], "CHIL"): # NB adoptive families i.e len(parent_families)>1 are not considered (TODO?)
                if member.get_pointer() == v.get_pointer():
                    continue
                siblings.add(term2id(member))

        if idx in individuals:
            children |= individuals[idx].get('children', set())
            siblings |= individuals[idx].get('siblings', set())
        individuals[idx] = {'sex': v.get_gender().lower(), 'children': children, 'siblings': siblings, 'title': title}

    elif isinstance(v,FamilyElement):
        wife, husb, children = None, None, set()
        children = set(term2id(i) for i in g.get_family_members(v, "CHIL"))

        try:
            wife = g.get_family_members(v, "WIFE")[0]
            wife = term2id(wife)
            if wife in individuals: individuals[wife]['children'] |= children
            else: individuals[wife] = {'children': children}
        except IndexError: pass
        try:
            husb = g.get_family_members(v, "HUSB")[0]
            husb = term2id(husb)
            if husb in individuals: individuals[husb]['children'] |= children
            else: individuals[husb] = {'children': children}
        except IndexError: pass

        if wife and husb: marriages[wife + husb] = (term2id(v), wife, husb)

for idx, val in individuals.items():
    added_terms = ''
    if val['sex'] == 'f':
        parent_predicate, sibl_predicate = "isMotherOf", "isSisterOf"
    else:
        parent_predicate, sibl_predicate = "isFatherOf", "isBrotherOf"
    if len(val['children']):
        added_terms += " ;\n    fhkb:" + parent_predicate + " " + ", ".join(["fhkb:" + i for i in val['children']])
    if len(val['siblings']):
        added_terms += " ;\n    fhkb:" + sibl_predicate + " " + ", ".join(["fhkb:" + i for i in val['siblings']])
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing%s ;\n    rdfs:label \"%s\" .\n" % (idx, added_terms, val['title']))

for k, v in marriages.items():
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing ;\n    fhkb:hasFemalePartner fhkb:%s ;\n    fhkb:hasMalePartner fhkb:%s .\n" % v)

out.write("[] a owl:AllDifferent ;\n    owl:distinctMembers (")
for idx in individuals.keys():
    out.write("    fhkb:" + idx)
for k, v in marriages.items():
    out.write("    fhkb:" + v[0])
out.write("    ) .")
out.close()

In [ ]:
!tail onto.ttl

### Inference karna

Ab hum is ontology ko inference aur querying ke liye use karna chahte hain. Hum [RDFLib](https://github.com/RDFLib) use karenge, jo RDF graphs ko alag-alag formats mein read karne, query karne, aadi ke liye ek library hai.

Logical inference ke liye, hum [OWL-RL](https://github.com/RDFLib/OWL-RL) library use karenge, jo humein RDF graph ka **Closure** banane deti hai, yaani woh saare possible concepts aur relations jodna jo infer kiye ja sakte hain.

In [ ]:
!{sys.executable} -m pip install rdflib
!{sys.executable} -m pip install git+https://github.com/RDFLib/OWL-RL.git

Chalo ontology file kholte hain aur dekhte hain ki isme kitne triplets hain:

In [ ]:
import rdflib
from owlrl import DeductiveClosure, OWLRL_Extension

g = rdflib.Graph()
g.parse("onto.ttl", format="turtle")

print("Triplets found:%d" % len(g))

In [ ]:
DeductiveClosure(OWLRL_Extension).expand(g)
print("Triplets after inference:%d" % len(g))

### Relatives ke liye query karna

Ab hum graph se query kar sakte hain taaki logon ke beech ke alag-alag relations dekh sakein. Hum `query` method ke saath **SPARQL** language use kar sakte hain. Hamare case mein, chalo apne family tree mein saare **uncles** dekhte hain:

In [ ]:
qres = g.query(
    """SELECT DISTINCT ?aname ?bname
       WHERE {
          ?a fhkb:isUncleOf ?b .
          ?a rdfs:label ?aname .
          ?b rdfs:label ?bname .
       }""")

for row in qres:
    print("%s is uncle of %s" % row)

Alag-alag family relations ke saath experiment karne mein sankoch na karein. Jaise, aap `isAncestorOf` relation dekh sakte hain, jo kisi diye gaye person ke saare ancestors ko recursively define karta hai.

Aakhir mein, chalo cleanup karte hain!

In [ ]:
!rm onto.ttl

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->